# NBA Polymarket Backtest (Parquet Version) — Colab

In [58]:
!pip install -q "numpy<2.0.0" "pandas==2.2.2" "scikit-learn==1.5.2" "xgboost==2.1.3" "nba-on-court>=0.2.0,<1.0" "joblib>=1.3" "matplotlib>=3.8" "tqdm>=4.66" "pyarrow"

import os
from google.colab import drive

# Mount Google Drive to save data persistently
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nba_backtest_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Data directory ready at: {DRIVE_DIR}")

# Clone repository and install local module
REPO_PATH = '/content/nba_bot'
if os.path.exists(REPO_PATH):
    !rm -rf {REPO_PATH}

!git clone https://github.com/cyruslayo/nba_bot.git {REPO_PATH}

!pip uninstall -y nba-bot > /dev/null 2>&1 || true
!pip install -e /content/nba_bot --no-deps -q

# Verify the fix was applied
import subprocess
result = subprocess.run(['grep', '-n', 'per_mode', '/content/nba_bot/nba_bot/team_stats_cache.py'], capture_output=True, text=True)
print(f"team_stats_cache.py per_mode parameter: {result.stdout.strip() if result.stdout.strip() else 'NOT FOUND (check install)'}")

print('Setup complete')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data directory ready at: /content/drive/MyDrive/nba_backtest_data
Cloning into '/content/nba_bot'...
remote: Enumerating objects: 517, done.
remote: Counting objects: 100% (517/517), done.
remote: Compressing objects: 100% (401/401), done.
remote: Total 517 (delta 139), reused 481 (delta 107), pack-reused 0 (from 0)
Receiving objects: 100% (517/517), 729.74 KiB | 1.08 MiB/s, done.
Resolving deltas: 100% (139/139), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nba-bot (pyproject.toml) ... done
team_stats_cache.py per_mode parameter: 52:            per_mode_detailed="PerGame",
Setup complete


In [59]:
import requests
from datetime import datetime, timedelta

def download_pmxt_parquet(start_date, end_date, save_dir):
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        date_str = current.strftime("%Y-%m-%d")

        # Loop through 24 hours (00 to 23)
        for hour in range(24):
            hour_str = f"{hour:02d}"
            filename = f"polymarket_orderbook_{date_str}T{hour_str}.parquet"
            url = f"https://archive.pmxt.dev/Polymarket/{filename}"
            save_path = os.path.join(save_dir, filename)

            if not os.path.exists(save_path):
                print(f"Downloading {filename}...")
                response = requests.get(url, stream=True)

                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"Saved to {save_path}")
                else:
                    pass # Skip missing hours without erroring
            else:
                pass # Already exists

        current += timedelta(days=1)
    print("Download process finished.")

# Example: Download data for a specific day/week and save it to your Drive (Uncomment to execute)
# download_pmxt_parquet("2024-01-01", "2024-01-02", DRIVE_DIR)

In [60]:
import json
import pandas as pd
import requests


def _safe_gamma_list(value):
    if isinstance(value, list):
        return value
    if value in (None, '', [], {}):
        return []
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def _as_utc_timestamp(value):
    ts = pd.to_datetime(value, utc=True, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    return ts


def _gamma_market_rows(events: list[dict]) -> list[dict]:
    rows = []
    for event in events:
        event_id = event.get('id')
        event_title = event.get('title')
        event_slug = event.get('slug')
        event_start_ts = _as_utc_timestamp(event.get('startDate'))
        for market in event.get('markets', []):
            outcomes = _safe_gamma_list(market.get('outcomes'))
            outcome_prices = _safe_gamma_list(market.get('outcomePrices'))
            clob_token_ids = _safe_gamma_list(market.get('clobTokenIds'))
            try:
                liquidity = float(market.get('liquidity', 0) or 0)
            except (TypeError, ValueError):
                liquidity = 0.0
            rows.append({
                'event_id': str(event_id) if event_id is not None else None,
                'event_title': event_title,
                'event_slug': event_slug,
                'event_start_ts': event_start_ts,
                'market_id': str(market.get('id')) if market.get('id') is not None else None,
                'question': market.get('question'),
                'outcomes': outcomes,
                'outcome_prices': outcome_prices,
                'clob_token_ids': clob_token_ids,
                'liquidity': liquidity,
                'active': market.get('active'),
                'closed': market.get('closed'),
            })
    return rows


# Find historical NBA markets
def search_nba_markets(start_date: str, end_date: str, limit: int = 100, closed: bool = True, active: bool = True, buffer_days: int = 1) -> pd.DataFrame:
    start_bound = pd.Timestamp(start_date)
    end_bound = pd.Timestamp(end_date)
    if start_bound.tzinfo is None:
        start_bound = start_bound.tz_localize('UTC')
    else:
        start_bound = start_bound.tz_convert('UTC')
    if end_bound.tzinfo is None:
        end_bound = end_bound.tz_localize('UTC')
    else:
        end_bound = end_bound.tz_convert('UTC')
    start_bound = start_bound.normalize() - pd.Timedelta(days=buffer_days)
    end_bound = end_bound.normalize() + pd.Timedelta(days=buffer_days + 1) - pd.Timedelta(microseconds=1)

    status_filters = []
    if closed:
        status_filters.append({'active': 'false', 'closed': 'true'})
    if active:
        status_filters.append({'active': 'true', 'closed': 'false'})
    if not status_filters:
        status_filters.append({})

    seen_market_ids: set[str] = set()
    all_events: list[dict] = []

    for status_filter in status_filters:
        offset = 0
        while True:
            params = {
                'series_id': 10345,
                'tag_id': 100639,
                'limit': limit,
                'offset': offset,
            }
            params.update(status_filter)
            response = requests.get('https://gamma-api.polymarket.com/events', params=params, timeout=20)
            response.raise_for_status()
            page_events = response.json()
            if not page_events:
                break
            all_events.extend(page_events)
            if len(page_events) < limit:
                break
            offset += limit

    markets_df = pd.DataFrame(_gamma_market_rows(all_events))
    if markets_df.empty:
        return markets_df

    markets_df = markets_df[markets_df['market_id'].notna()].copy()
    markets_df['event_start_ts'] = pd.to_datetime(markets_df['event_start_ts'], utc=True, errors='coerce')
    markets_df = markets_df[markets_df['event_start_ts'].notna()].copy()
    markets_df = markets_df[(markets_df['event_start_ts'] >= start_bound) & (markets_df['event_start_ts'] <= end_bound)].copy()

    deduped_rows = []
    for row in markets_df.sort_values(['event_start_ts', 'market_id']).to_dict('records'):
        market_id = str(row['market_id'])
        if market_id in seen_market_ids:
            continue
        seen_market_ids.add(market_id)
        deduped_rows.append(row)

    return pd.DataFrame(deduped_rows).reset_index(drop=True)

In [61]:
import ast
import json
import math
import os
import re
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pyarrow.parquet as pq

from nba_bot.features import STARTTYPE_FALLBACK, _encode_starttype, build_game_state_rows
from nba_bot.model import load_model, predict_home_win_prob

MODEL_PATH = '/content/drive/MyDrive/nba_bot/xgb_model_t2.pkl'
PMXT_ROOT = Path(DRIVE_DIR)
PMXT_GLOB = '*.parquet'
NBA_SEASONS = [2024]
AUTO_INFER_NBA_SEASONS = True
USE_ADVANCED_FEATURES = True
INITIAL_BANKROLL = 1000.0
LATENCY_BUFFER_MS = 500
COOLDOWN_MS = 1500
MIN_TARGET_DELTA_USDC = 5.0
MIN_DIVERGENCE_RATIO = 0.10
MIN_RESTING_DEPTH_USDC = 50.0
ADVERSE_SELECTION_PENALTY = 0.01
FEE_RATE = 0.02
PER_GAME_BANKROLL_DIVISOR = 5.0
HOLDOUT_PCT = 0.30
CHUNK_SIZE = 100000
MARKET_DATE_BUFFER_DAYS = 1
MATCH_SCORE_MIN = 8.0
MATCH_SCORE_GAP_MIN = 2.5
MAX_TIPOFF_DELTA_HOURS = 18.0
LOOKUP_CACHE_DIR = Path(DRIVE_DIR) / 'market_lookup_cache'
LOOKUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REUSE_SAVED_LOOKUP = True
MARKET_ONLY_MODE = True

NBA_TEAM_DATA = {
    1610612737: {'city': 'Atlanta', 'team': 'Hawks', 'abbr': 'ATL', 'aliases': ['atlanta hawks', 'hawks', 'atlanta', 'atl']},
    1610612738: {'city': 'Boston', 'team': 'Celtics', 'abbr': 'BOS', 'aliases': ['boston celtics', 'celtics', 'boston', 'bos']},
    1610612751: {'city': 'Brooklyn', 'team': 'Nets', 'abbr': 'BKN', 'aliases': ['brooklyn nets', 'nets', 'brooklyn', 'bkn']},
    1610612766: {'city': 'Charlotte', 'team': 'Hornets', 'abbr': 'CHA', 'aliases': ['charlotte hornets', 'hornets', 'charlotte', 'cha']},
    1610612741: {'city': 'Chicago', 'team': 'Bulls', 'abbr': 'CHI', 'aliases': ['chicago bulls', 'bulls', 'chicago', 'chi']},
    1610612739: {'city': 'Cleveland', 'team': 'Cavaliers', 'abbr': 'CLE', 'aliases': ['cleveland cavaliers', 'cavaliers', 'cavs', 'cleveland', 'cle']},
    1610612742: {'city': 'Dallas', 'team': 'Mavericks', 'abbr': 'DAL', 'aliases': ['dallas mavericks', 'mavericks', 'mavs', 'dallas', 'dal']},
    1610612743: {'city': 'Denver', 'team': 'Nuggets', 'abbr': 'DEN', 'aliases': ['denver nuggets', 'nuggets', 'denver', 'den']},
    1610612765: {'city': 'Detroit', 'team': 'Pistons', 'abbr': 'DET', 'aliases': ['detroit pistons', 'pistons', 'detroit', 'det']},
    1610612744: {'city': 'Golden State', 'team': 'Warriors', 'abbr': 'GSW', 'aliases': ['golden state warriors', 'warriors', 'golden state', 'gsw']},
    1610612745: {'city': 'Houston', 'team': 'Rockets', 'abbr': 'HOU', 'aliases': ['houston rockets', 'rockets', 'houston', 'hou']},
    1610612754: {'city': 'Indiana', 'team': 'Pacers', 'abbr': 'IND', 'aliases': ['indiana pacers', 'pacers', 'indiana', 'ind']},
    1610612746: {'city': 'Los Angeles', 'team': 'Clippers', 'abbr': 'LAC', 'aliases': ['los angeles clippers', 'la clippers', 'clippers', 'lac']},
    1610612747: {'city': 'Los Angeles', 'team': 'Lakers', 'abbr': 'LAL', 'aliases': ['los angeles lakers', 'la lakers', 'lakers', 'lal']},
    1610612763: {'city': 'Memphis', 'team': 'Grizzlies', 'abbr': 'MEM', 'aliases': ['memphis grizzlies', 'grizzlies', 'memphis', 'mem']},
    1610612748: {'city': 'Miami', 'team': 'Heat', 'abbr': 'MIA', 'aliases': ['miami heat', 'heat', 'miami', 'mia']},
    1610612749: {'city': 'Milwaukee', 'team': 'Bucks', 'abbr': 'MIL', 'aliases': ['milwaukee bucks', 'bucks', 'milwaukee', 'mil']},
    1610612750: {'city': 'Minnesota', 'team': 'Timberwolves', 'abbr': 'MIN', 'aliases': ['minnesota timberwolves', 'timberwolves', 'wolves', 'minnesota', 'min']},
    1610612740: {'city': 'New Orleans', 'team': 'Pelicans', 'abbr': 'NOP', 'aliases': ['new orleans pelicans', 'pelicans', 'new orleans', 'nop', 'nola']},
    1610612752: {'city': 'New York', 'team': 'Knicks', 'abbr': 'NYK', 'aliases': ['new york knicks', 'knicks', 'new york', 'nyk']},
    1610612760: {'city': 'Oklahoma City', 'team': 'Thunder', 'abbr': 'OKC', 'aliases': ['oklahoma city thunder', 'thunder', 'oklahoma city', 'okc']},
    1610612753: {'city': 'Orlando', 'team': 'Magic', 'abbr': 'ORL', 'aliases': ['orlando magic', 'magic', 'orlando', 'orl']},
    1610612755: {'city': 'Philadelphia', 'team': '76ers', 'abbr': 'PHI', 'aliases': ['philadelphia 76ers', '76ers', 'sixers', 'philadelphia', 'phi']},
    1610612756: {'city': 'Phoenix', 'team': 'Suns', 'abbr': 'PHX', 'aliases': ['phoenix suns', 'suns', 'phoenix', 'phx']},
    1610612757: {'city': 'Portland', 'team': 'Trail Blazers', 'abbr': 'POR', 'aliases': ['portland trail blazers', 'trail blazers', 'blazers', 'portland', 'por']},
    1610612758: {'city': 'Sacramento', 'team': 'Kings', 'abbr': 'SAC', 'aliases': ['sacramento kings', 'kings', 'sacramento', 'sac']},
    1610612759: {'city': 'San Antonio', 'team': 'Spurs', 'abbr': 'SAS', 'aliases': ['san antonio spurs', 'spurs', 'san antonio', 'sas']},
    1610612761: {'city': 'Toronto', 'team': 'Raptors', 'abbr': 'TOR', 'aliases': ['toronto raptors', 'raptors', 'toronto', 'tor']},
    1610612762: {'city': 'Utah', 'team': 'Jazz', 'abbr': 'UTA', 'aliases': ['utah jazz', 'jazz', 'utah', 'uta']},
    1610612764: {'city': 'Washington', 'team': 'Wizards', 'abbr': 'WAS', 'aliases': ['washington wizards', 'wizards', 'washington', 'was']},
}

MARKET_TO_GAME: dict[str, Any] = {}
MARKET_LOOKUP = pd.DataFrame()
MARKET_MATCHES = pd.DataFrame()
MATCH_DIAGNOSTICS: dict[str, pd.DataFrame] = {}

BACKTEST_STATE_PATH = Path('/content/backtest_state.json')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [69]:
def build_game_catalog_from_gamma(markets_df: pd.DataFrame) -> pd.DataFrame:
    """Build a game catalog from Gamma API market metadata instead of NBA stats."""
    if markets_df.empty:
        return pd.DataFrame()

    catalog_rows = []
    seen_games = set()

    for row in markets_df.to_dict('records'):
        market_id = str(row.get('market_id') or '')
        event_title = str(row.get('event_title') or '')
        question = str(row.get('question') or '')
        event_start_ts = row.get('event_start_ts')
        outcomes = row.get('outcomes', [])

        if not market_id or not event_title:
            continue

        game_id = f"GAMMA_{market_id}"
        if game_id in seen_games:
            continue
        seen_games.add(game_id)

        home_team, away_team = None, None
        yes_team_side = 'home'

        title_lower = event_title.lower()
        question_lower = question.lower()

        if ' vs ' in title_lower:
            parts = title_lower.split(' vs ')
            if len(parts) == 2:
                away_team = parts[0].strip().title()
                home_team = parts[1].strip().title()
        elif ' vs ' in question_lower:
            parts = question_lower.split(' vs ')
            if len(parts) == 2:
                away_team = parts[0].strip().title()
                home_team = parts[1].strip().title()
        elif ' @ ' in title_lower:
            parts = title_lower.split(' @ ')
            if len(parts) == 2:
                away_team = parts[1].strip().title()
                home_team = parts[0].strip().title()

        if outcomes and len(outcomes) >= 2:
            outcome_0 = str(outcomes[0]).lower()
            outcome_1 = str(outcomes[1]).lower()

            if home_team:
                home_lower = home_team.lower()
                away_lower = away_team.lower() if away_team else ''

                if home_lower in outcome_0 or (home_team and home_team.lower() in outcome_0):
                    yes_team_side = 'home'
                elif away_lower and (away_lower in outcome_0 or away_team.lower() in outcome_0):
                    yes_team_side = 'away'
                elif home_lower in outcome_1 or (home_team and home_team.lower() in outcome_1):
                    yes_team_side = 'away'
                elif away_lower and (away_lower in outcome_1 or away_team.lower() in outcome_1):
                    yes_team_side = 'home'

        game_date = None
        if pd.notna(event_start_ts):
            game_date = pd.Timestamp(event_start_ts).normalize()

        catalog_rows.append({
            'game_id': game_id,
            'game_date': game_date,
            'home_team': home_team or 'Unknown',
            'away_team': away_team or 'Unknown',
            'tipoff_time': event_start_ts,
            'market_id': market_id,
            'event_title': event_title,
            'question': question,
            'yes_team_side': yes_team_side,
            'liquidity': float(row.get('liquidity') or 0),
        })

    return pd.DataFrame(catalog_rows)


def build_simple_timeline_from_gamma(markets_df: pd.DataFrame) -> pd.DataFrame:
    """Build a minimal timeline for market-only mode without NBA stats."""
    timeline_rows = []

    for row in markets_df.to_dict('records'):
        market_id = str(row.get('market_id') or '')
        event_start_ts = row.get('event_start_ts')

        if not market_id or pd.isna(event_start_ts):
            continue

        game_id = f"GAMMA_{market_id}"
        tipoff = pd.Timestamp(event_start_ts)

        timeline_rows.append({
            'game_id': game_id,
            'event_time': tipoff,
            'period': 1,
            'clock': '12:00',
            'home_score': 0.0,
            'away_score': 0.0,
            'time_remaining': 2880.0,
            'starttype_encoded': STARTTYPE_FALLBACK,
            'player_quality_home': 0.0,
            'player_quality_away': 0.0,
            'latent_time': tipoff + pd.to_timedelta(LATENCY_BUFFER_MS, unit='ms'),
            'tipoff_time': tipoff,
        })

    df = pd.DataFrame(timeline_rows)
    if not df.empty:
        df['latent_time'] = pd.to_datetime(df['latent_time'], utc=True).dt.as_unit('ns')
        df['tipoff_time'] = pd.to_datetime(df['tipoff_time'], utc=True).dt.as_unit('ns')
        df = df.sort_values(['game_id', 'latent_time']).reset_index(drop=True)
    return df


def join_ticks_with_timeline(frame: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    ticks = frame.copy()
    ticks['timestamp'] = pd.to_datetime(ticks['timestamp'], utc=True, errors='coerce').dt.as_unit('ns')
    ticks['game_id'] = ticks['game_id'].astype(str)
    ticks = ticks[ticks['timestamp'].notna()].sort_values(['game_id', 'timestamp']).reset_index(drop=True)

    timeline_fixed = timeline.copy()
    timeline_fixed['latent_time'] = pd.to_datetime(timeline_fixed['latent_time'], utc=True).dt.as_unit('ns')
    timeline_fixed = timeline_fixed.sort_values(['game_id', 'latent_time']).reset_index(drop=True)

    merged = pd.merge_asof(
        ticks,
        timeline_fixed,
        left_on='timestamp',
        right_on='latent_time',
        by='game_id',
        direction='backward',
        allow_exact_matches=True,
    )
    return merged[merged['timestamp'] >= merged['tipoff_time']].copy()


def attach_game_ids(frame: pd.DataFrame, market_lookup: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    key_col = _candidate_column(list(out.columns), ['market_id', 'slug', 'event_slug', 'market'])
    if key_col is None:
        return pd.DataFrame()
    
    out['market_id_raw'] = out[key_col].astype(str)
    
    lookup_dict = dict(zip(market_lookup['lookup_key'].astype(str), market_lookup['game_id'].astype(str)))
    yes_team_dict = dict(zip(market_lookup['lookup_key'].astype(str), market_lookup['yes_team_side'].astype(str)))
    
    out['game_id'] = out['market_id_raw'].map(lookup_dict)
    out['yes_team_side'] = out['market_id_raw'].map(yes_team_dict)
    out['resolved_market_id'] = out['market_id_raw']
    
    return out[out['game_id'].notna()].copy()


def infer_pmxt_date_range(paths: list[Path]) -> tuple[str | None, str | None]:
    """Infer date range from PMXT parquet filenames."""
    import re
    dates = set()
    pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
    for p in paths:
        match = pattern.search(p.name)
        if match:
            dates.add(match.group(1))
    if not dates:
        return None, None
    sorted_dates = sorted(dates)
    return sorted_dates[0], sorted_dates[-1]


def _candidate_column(columns: list[str], names: list[str]) -> str | None:
    lowered = {str(c).lower(): c for c in columns}
    for name in names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    for column in columns:
        if any(name.lower() in str(column).lower() for name in names):
            return column
    return None


def _safe_json(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except Exception:
            try:
                return ast.literal_eval(text)
            except Exception:
                return None
    return None

In [63]:
def parse_levels(raw_levels: Any, descending: bool) -> list[tuple[float, float]]:
    payload = _safe_json(raw_levels)
    normalized = []
    for level in payload or []:
        price, size = None, None
        if isinstance(level, dict):
            price = level.get('price') or level.get('px') or level.get('p')
            size = level.get('size') or level.get('sz') or level.get('quantity') or level.get('q')
        elif isinstance(level, (list, tuple)) and len(level) >= 2:
            price, size = level[0], level[1]
        try:
            price, size = float(price), float(size)
        except (TypeError, ValueError):
            continue
        if price <= 0 or price >= 1 or size <= 0: continue
        normalized.append((float(price), float(size)))
    normalized.sort(key=lambda item: item[0], reverse=descending)
    return normalized

class OrderBookState:
    def __init__(self) -> None:
        self.bids: dict[float, float] = {}
        self.asks: dict[float, float] = {}

    def _apply_side(self, side: str, levels: list[tuple[float, float]], replace: bool) -> None:
        book = self.bids if side == 'bids' else self.asks
        if replace: book.clear()
        for price, size in levels:
            if size <= 0: book.pop(price, None)
            else: book[float(price)] = float(size)

    def update(self, record: dict[str, Any]) -> None:
        event_type = str(record.get('event_type') or record.get('type') or '').lower()
        bids = parse_levels(record.get('bids'), descending=True)
        asks = parse_levels(record.get('asks'), descending=False)
        is_snapshot = event_type == 'book_snapshot'

        if bids: self._apply_side('bids', bids, replace=is_snapshot)
        elif is_snapshot: self.bids.clear()
        if asks: self._apply_side('asks', asks, replace=is_snapshot)
        elif is_snapshot: self.asks.clear()

        for change in _safe_json(record.get('price_change')) or []:
            if isinstance(change, dict):
                side = str(change.get('side') or change.get('book') or '').lower()
                price = change.get('price') or change.get('px')
                size = change.get('size') or change.get('sz') or 0
            elif isinstance(change, (list, tuple)) and len(change) >= 3:
                side, price, size = change[0], change[1], change[2]
            else:
                continue
            try:
                price, size = float(price), float(size)
            except (TypeError, ValueError): continue

            target = self.bids if 'bid' in side else self.asks
            if size <= 0: target.pop(price, None)
            else: target[price] = size

    def best_bid(self) -> tuple[float | None, float]:
        if not self.bids: return None, 0.0
        price = max(self.bids)
        return price, float(self.bids[price])

    def best_ask(self) -> tuple[float | None, float]:
        if not self.asks: return None, 0.0
        price = min(self.asks)
        return price, float(self.asks[price])

    def vwap_buy(self, stake_usdc: float) -> tuple[float | None, float]:
        remaining, total_cost, total_shares = float(stake_usdc), 0.0, 0.0
        for price in sorted(self.asks):
            size = float(self.asks[price])
            if size <= 0: continue
            level_cost = float(price) * size
            if level_cost <= remaining:
                total_cost += level_cost
                total_shares += size
                remaining -= level_cost
            else:
                shares = remaining / float(price)
                total_cost += remaining
                total_shares += shares
                remaining = 0.0
                break
        if total_shares <= 0 or remaining > 1e-6: return None, total_shares
        return total_cost / total_shares, total_shares

@dataclass
class MarketPosition:
    market_id: str
    game_id: Any
    yes_shares: float = 0.0
    no_shares: float = 0.0
    yes_cost: float = 0.0
    no_cost: float = 0.0
    realized_pnl: float = 0.0
    cooldown_until: pd.Timestamp | None = None
    settled: bool = False
    settlement_value: float | None = None

@dataclass
class PortfolioState:
    initial_bankroll: float
    cash: float
    realized_pnl: float = 0.0
    positions: dict[str, MarketPosition] = field(default_factory=dict)

def total_bankroll(portfolio: PortfolioState) -> float:
    return float(portfolio.initial_bankroll + portfolio.realized_pnl)

def available_capital(portfolio: PortfolioState) -> float:
    return max(total_bankroll(portfolio) / PER_GAME_BANKROLL_DIVISOR, 0.0)

def fee_adjusted_kelly(probability: float, market_price: float, fee_rate: float = FEE_RATE, fraction: float = 0.25) -> float:
    if not (0 < market_price < 1) or not (0 < probability < 1): return 0.0
    b = (1.0 - market_price) / market_price
    q = 1.0 - probability
    full_kelly = ((probability * (1.0 - fee_rate) * b) - q) / b
    return round(max(full_kelly * fraction, 0.0), 6)

def ensure_position(portfolio: PortfolioState, market_id: str, game_id: Any) -> MarketPosition:
    if market_id not in portfolio.positions:
        portfolio.positions[market_id] = MarketPosition(market_id=market_id, game_id=game_id)
    return portfolio.positions[market_id]

def merge_positions(portfolio: PortfolioState, position: MarketPosition) -> None:
    paired = min(position.yes_shares, position.no_shares)
    if paired <= 0: return
    yes_release_cost = position.yes_cost * (paired / position.yes_shares) if position.yes_shares > 0 else 0.0
    no_release_cost = position.no_cost * (paired / position.no_shares) if position.no_shares > 0 else 0.0
    position.yes_shares -= paired
    position.no_shares -= paired
    position.yes_cost -= yes_release_cost
    position.no_cost -= no_release_cost
    realized = paired - yes_release_cost - no_release_cost
    portfolio.cash += paired
    portfolio.realized_pnl += realized
    position.realized_pnl += realized

def execute_buy(portfolio: PortfolioState, position: MarketPosition, book: OrderBookState, side: str, stake_usdc: float) -> dict[str, float] | None:
    if stake_usdc <= 0: return None
    spend = min(float(stake_usdc), float(portfolio.cash))
    if spend <= 0: return None
    raw_vwap, raw_shares = book.vwap_buy(spend)
    if raw_vwap is None or raw_shares <= 0: return None
    effective_price = min(max(float(raw_vwap) + ADVERSE_SELECTION_PENALTY, 0.001), 0.999)
    shares = spend / effective_price
    portfolio.cash -= spend
    if side == 'YES':
        position.yes_shares += shares
        position.yes_cost += spend
    else:
        position.no_shares += shares
        position.no_cost += spend
    merge_positions(portfolio, position)
    return {'spend': spend, 'price': effective_price, 'shares': shares}

def settle_position(portfolio: PortfolioState, position: MarketPosition, outcome_yes: bool) -> dict[str, float] | None:
    if position.settled: return None
    winning_shares = position.yes_shares if outcome_yes else position.no_shares
    total_cost = position.yes_cost + position.no_cost
    proceeds = float(winning_shares)
    gross_profit = proceeds - total_cost
    fee = max(gross_profit, 0.0) * FEE_RATE
    net_profit = gross_profit - fee
    portfolio.cash += proceeds - fee
    portfolio.realized_pnl += net_profit
    position.realized_pnl += net_profit
    position.yes_shares, position.no_shares, position.yes_cost, position.no_cost = 0.0, 0.0, 0.0, 0.0
    position.settled = True
    position.settlement_value = 1.0 if outcome_yes else 0.0
    return {'proceeds': proceeds, 'gross_profit': gross_profit, 'fee': fee, 'net_profit': net_profit}

In [64]:
def normalize_pmxt_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c) for c in out.columns]

    timestamp_col = _candidate_column(list(out.columns), ['timestamp', 'event_time', 'ts', 'created_at'])
    market_col = _candidate_column(list(out.columns), ['market_id', 'market', 'slug', 'event_slug'])
    event_col = _candidate_column(list(out.columns), ['event_type', 'type'])
    slug_col = _candidate_column(list(out.columns), ['slug'])
    event_slug_col = _candidate_column(list(out.columns), ['event_slug'])
    market_alias_col = _candidate_column(list(out.columns), ['market'])

    if timestamp_col is None or market_col is None:
        raise ValueError('PMXT chunk is missing a timestamp or market identifier column')

    out['timestamp'] = pd.to_datetime(out[timestamp_col], utc=True, errors='coerce')
    out['market_id'] = out[market_col].astype(str)
    out['event_type'] = out[event_col].astype(str) if event_col is not None else 'price_change'
    if slug_col is not None:
        out['slug'] = out[slug_col].astype(str)
    if event_slug_col is not None:
        out['event_slug'] = out[event_slug_col].astype(str)
    if market_alias_col is not None:
        out['market'] = out[market_alias_col].astype(str)

    for col in ['bids', 'asks', 'price_change', 'resolution']:
        if col not in out.columns: out[col] = None
    return out


def iter_pmxt_chunks(paths: list[Path], chunksize: int = CHUNK_SIZE):
    for path in paths:
        parquet_file = pq.ParquetFile(path)
        for batch in parquet_file.iter_batches(batch_size=chunksize):
            yield path, batch.to_pandas()


def _extract_resolution(record: dict[str, Any]) -> bool | None:
    event_type = str(record.get('event_type') or '').lower()
    if event_type not in {'condition_resolved', 'market_resolution'} and record.get('resolution') in (None, '', {}, []):
        return None
    resolution = record.get('resolution')
    payload = _safe_json(resolution) if not isinstance(resolution, dict) else resolution
    winner = str(payload.get('winner') or payload.get('outcome') or payload.get('result') or '').lower() if isinstance(payload, dict) else str(resolution).lower()
    if winner in {'yes', '1', 'true', 'home'}: return True
    if winner in {'no', '0', 'false', 'away'}: return False
    return None


def evaluate_signal(probability_yes: float, position: MarketPosition, portfolio: PortfolioState, book: OrderBookState) -> tuple[str, float] | None:
    best_ask, ask_size = book.best_ask()
    best_bid, bid_size = book.best_bid()
    if best_ask is None or best_bid is None: return None

    if float(best_ask) * float(ask_size) < MIN_RESTING_DEPTH_USDC or float(best_bid) * float(bid_size) < MIN_RESTING_DEPTH_USDC:
        return None

    yes_fraction = fee_adjusted_kelly(probability_yes, float(best_ask))
    no_fraction = fee_adjusted_kelly(1.0 - probability_yes, max(min(1.0 - float(best_bid), 0.999), 0.001))

    if yes_fraction <= 0 and no_fraction <= 0: return None

    if yes_fraction >= no_fraction:
        target = available_capital(portfolio) * yes_fraction
        delta = target - position.yes_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'YES', float(delta)
    else:
        target = available_capital(portfolio) * no_fraction
        delta = target - position.no_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'NO', float(delta)
    return None


def split_paths(paths: list[Path], holdout_pct: float = HOLDOUT_PCT) -> tuple[list[Path], list[Path]]:
    ordered = sorted(paths)
    if not ordered: return [], []
    cutoff = min(max(1, int(len(ordered) * (1.0 - holdout_pct))), len(ordered))
    return ordered[:cutoff], ordered[cutoff:]


def run_backtest(paths: list[Path], timeline: pd.DataFrame, market_lookup: pd.DataFrame, model: Any, feature_cols: list[str] | None, initial_bankroll: float = INITIAL_BANKROLL) -> dict[str, Any]:
    portfolio = PortfolioState(initial_bankroll=initial_bankroll, cash=initial_bankroll)
    books: dict[str, OrderBookState] = {}
    trades, equity = [], []

    for path, chunk in iter_pmxt_chunks(paths):
        joined = join_ticks_with_timeline(attach_game_ids(normalize_pmxt_frame(chunk), market_lookup), timeline)

        for record in joined.sort_values('timestamp').to_dict('records'):
            market_id = str(record.get('resolved_market_id') or record['market_id'])
            position = ensure_position(portfolio, market_id, record['game_id'])
            book = books.setdefault(market_id, OrderBookState())
            book.update(record)

            resolution = _extract_resolution(record)
            if resolution is not None:
                settlement = settle_position(portfolio, position, resolution)
                if settlement:
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': 'SETTLE_YES' if resolution else 'SETTLE_NO', **settlement})
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.settled or (record.get('time_remaining') is not None and float(record['time_remaining']) <= 0):
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.cooldown_until is not None and pd.Timestamp(record['timestamp']) < position.cooldown_until:
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            adv_ctx = None
            if USE_ADVANCED_FEATURES:
                adv_ctx = {'starttype_encoded': int(record.get('starttype_encoded', STARTTYPE_FALLBACK) or STARTTYPE_FALLBACK), 'player_quality_home': float(record.get('player_quality_home', 0.0) or 0.0), 'player_quality_away': float(record.get('player_quality_away', 0.0) or 0.0)}

            prob_home = predict_home_win_prob(model=model, feature_cols=feature_cols, home_score=int(record['home_score']), away_score=int(record['away_score']), period=int(record['period']), clock=str(record['clock']), advanced_ctx=adv_ctx)
            yes_team_side = str(record.get('yes_team_side') or 'home').lower()
            prob_yes = prob_home if yes_team_side == 'home' else 1.0 - prob_home

            signal = evaluate_signal(prob_yes, position, portfolio, book)
            if signal:
                side, delta = signal
                fill = execute_buy(portfolio, position, book, side, delta)
                if fill:
                    position.cooldown_until = pd.Timestamp(record['timestamp']) + pd.to_timedelta(COOLDOWN_MS, unit='ms')
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': f'BUY_{side}', 'model_prob_home': prob_home, 'model_prob_yes': prob_yes, 'yes_team_side': yes_team_side, 'cash_after': portfolio.cash, **fill})
            equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})

    BACKTEST_STATE_PATH.write_text(json.dumps({'initial_bankroll': portfolio.initial_bankroll, 'cash': portfolio.cash, 'realized_pnl': portfolio.realized_pnl, 'positions': {m: asdict(p) for m, p in portfolio.positions.items()}}, default=str, indent=2), encoding='utf-8')
    return {'portfolio': portfolio, 'trades': pd.DataFrame(trades), 'equity_curve': pd.DataFrame(equity), 'state_path': str(BACKTEST_STATE_PATH)}


def plot_equity_curve(result: dict[str, Any], label: str) -> None:
    equity = result['equity_curve'].copy()
    if equity.empty: return
    sampled = equity.iloc[::max(len(equity) // min(2000, len(equity)), 1)].copy()
    plt.figure(figsize=(12, 4))
    plt.plot(sampled['timestamp'], sampled['equity'])
    plt.title(f'Equity Curve — {label}')
    plt.xlabel('Timestamp')
    plt.ylabel('USDC')
    plt.grid(True, alpha=0.3)
    plt.show()

In [70]:
model, feature_cols = load_model(MODEL_PATH)
if model is None: raise FileNotFoundError(f'Model not found at {MODEL_PATH}')

pmxt_paths = sorted(PMXT_ROOT.glob(PMXT_GLOB))
if not pmxt_paths: raise FileNotFoundError(f'No PMXT files found under {PMXT_ROOT} matching {PMXT_GLOB}')

pmxt_start_date, pmxt_end_date = infer_pmxt_date_range(pmxt_paths)
print(f'PMXT date range: {pmxt_start_date} -> {pmxt_end_date}')

if MARKET_ONLY_MODE:
    print('Running in MARKET-ONLY mode - using Gamma API as game catalog (no NBA stats required)')

    gamma_markets_df = search_nba_markets(pmxt_start_date, pmxt_end_date, limit=100, closed=True, active=True, buffer_days=MARKET_DATE_BUFFER_DAYS)
    if gamma_markets_df.empty:
        raise ValueError(f'No Gamma NBA markets found between {pmxt_start_date} and {pmxt_end_date}')

    print(f'Found {len(gamma_markets_df)} Gamma markets')

    game_catalog = build_game_catalog_from_gamma(gamma_markets_df)
    if game_catalog.empty:
        raise ValueError('Could not build game catalog from Gamma markets')

    print(f'Built game catalog with {len(game_catalog)} games')

    timeline = build_simple_timeline_from_gamma(gamma_markets_df)
    if timeline.empty:
        raise ValueError('Could not build timeline from Gamma markets')

    print(f'Built timeline with {len(timeline)} entries')

    MARKET_MATCHES = game_catalog.copy()
    MARKET_MATCHES['score'] = 10.0
    MARKET_MATCHES['score_gap'] = 5.0
    MARKET_MATCHES['time_delta_hours'] = 0.0

    MARKET_LOOKUP = pd.DataFrame({
        'lookup_key': game_catalog['market_id'].astype(str),
        'game_id': game_catalog['game_id'].astype(str),
        'yes_team_side': game_catalog['yes_team_side'].astype(str),
        'market_id': game_catalog['market_id'].astype(str),
        'event_slug': game_catalog.get('event_slug', pd.NA),
    })

    MATCH_DIAGNOSTICS = {
        'unmatched_games': pd.DataFrame(),
        'unmatched_markets': pd.DataFrame(),
        'ambiguous_candidates': pd.DataFrame(),
    }

    print(f'Market lookup has {len(MARKET_LOOKUP)} entries')
else:
    def _season_for_date(value: str | pd.Timestamp) -> int:
        ts = pd.Timestamp(value)
        return ts.year if ts.month >= 10 else ts.year - 1

    active_seasons = list(NBA_SEASONS)
    if AUTO_INFER_NBA_SEASONS and pmxt_start_date is not None and pmxt_end_date is not None:
        start_ts = pd.Timestamp(pmxt_start_date)
        end_ts = pd.Timestamp(pmxt_end_date)
        active_seasons = sorted({_season_for_date(start_ts), _season_for_date(end_ts)})

    print(f'Using NBA seasons: {active_seasons}')

    df_nbastats = load_data_safe(active_seasons, 'nbastats')
    if df_nbastats is None or df_nbastats.empty:
        raise ValueError(f'nbastats data could not be loaded for seasons {active_seasons}')

    df_pbpstats = load_data_safe(active_seasons, 'pbpstats') if USE_ADVANCED_FEATURES else None
    player_ratings = load_player_ratings(USE_ADVANCED_FEATURES)
    timeline = prepare_pbp_timeline(df_nbastats, df_pbpstats, player_ratings)
    game_catalog = build_game_catalog(df_nbastats)

    if pmxt_start_date is None or pmxt_end_date is None:
        dated_games = game_catalog['game_date'].dropna()
        if dated_games.empty:
            raise ValueError('Unable to infer a date range from PMXT files or game catalog')
        pmxt_start_date = dated_games.min().strftime('%Y-%m-%d')
        pmxt_end_date = dated_games.max().strftime('%Y-%m-%d')

    game_catalog = game_catalog.copy()
    game_catalog['game_date'] = pd.to_datetime(game_catalog['game_date'], errors='coerce').dt.normalize()

    lower_game_date = pd.Timestamp(pmxt_start_date).normalize() - pd.Timedelta(days=MARKET_DATE_BUFFER_DAYS)
    upper_game_date = pd.Timestamp(pmxt_end_date).normalize() + pd.Timedelta(days=MARKET_DATE_BUFFER_DAYS)
    relevant_games = game_catalog[(game_catalog['game_date'].notna()) & (game_catalog['game_date'] >= lower_game_date) & (game_catalog['game_date'] <= upper_game_date)].copy()
    if relevant_games.empty:
        available_dates = game_catalog['game_date'].dropna()
        coverage_msg = 'no valid game dates were loaded'
        if not available_dates.empty:
            coverage_msg = f"loaded game catalog covers {available_dates.min().strftime('%Y-%m-%d')} -> {available_dates.max().strftime('%Y-%m-%d')}"
        raise ValueError(
            f'No games in the backtest catalog fall within the PMXT date range {lower_game_date.strftime("%Y-%m-%d")} -> {upper_game_date.strftime("%Y-%m-%d")}; {coverage_msg}. '
            f'Check that the PMXT files and loaded NBA seasons align.'
        )

    cache_key = build_lookup_cache_key(pmxt_start_date, pmxt_end_date, active_seasons, len(pmxt_paths))
    cache_paths = lookup_cache_paths(cache_key)
    print(f'Lookup cache key: {cache_key}')

    cached_lookup = cached_matches = cached_diagnostics = None
    if REUSE_SAVED_LOOKUP:
        cached_lookup, cached_matches, cached_diagnostics = load_lookup_cache(cache_key)

    if cached_lookup is not None and cached_matches is not None and cached_diagnostics is not None:
        MARKET_LOOKUP = cached_lookup
        MARKET_MATCHES = cached_matches
        MATCH_DIAGNOSTICS = cached_diagnostics
        print(f'Loaded cached market lookup from: {cache_paths["lookup"]}')
    else:
        gamma_markets_df = search_nba_markets(pmxt_start_date, pmxt_end_date, limit=100, closed=True, active=True, buffer_days=MARKET_DATE_BUFFER_DAYS)
        if gamma_markets_df.empty:
            raise ValueError(f'No Gamma NBA markets found between {pmxt_start_date} and {pmxt_end_date}')

        MARKET_MATCHES, MATCH_DIAGNOSTICS = match_gamma_markets_to_games(gamma_markets_df, relevant_games, buffer_days=MARKET_DATE_BUFFER_DAYS)
        if MARKET_MATCHES.empty:
            print('No strict market matches found. Showing diagnostics...')
            display(MATCH_DIAGNOSTICS.get('unmatched_markets', pd.DataFrame()).head(20))
            display(MATCH_DIAGNOSTICS.get('unmatched_games', pd.DataFrame()).head(20))
            display(MATCH_DIAGNOSTICS.get('ambiguous_candidates', pd.DataFrame()).head(20))
            raise ValueError('No Gamma markets could be matched to the backtest game catalog for the requested date range')

        MARKET_LOOKUP = build_market_lookup(MARKET_MATCHES)
        save_lookup_cache(cache_key, MARKET_LOOKUP, MARKET_MATCHES, MATCH_DIAGNOSTICS)
        print(f'Saved market lookup cache to: {cache_paths["lookup"]}')

print(f"Matched {MARKET_MATCHES['market_id'].nunique() if 'market_id' in MARKET_MATCHES.columns else len(MARKET_MATCHES)} markets to games")
display_cols = ['game_date', 'game_id', 'home_team', 'away_team', 'market_id', 'event_title', 'question', 'yes_team_side']
available_cols = [col for col in display_cols if col in MARKET_MATCHES.columns]
if available_cols:
    display(MARKET_MATCHES[available_cols].head(20))

print(f"Beginning Backtest Iteration on {len(pmxt_paths)} files")
tuning_result = run_backtest(pmxt_paths, timeline, MARKET_LOOKUP, model, feature_cols, initial_bankroll=INITIAL_BANKROLL)

print(f"Ending Balance: {tuning_result['portfolio'].cash:.2f}")
plot_equity_curve(tuning_result, 'full')

PMXT date range: 2026-02-21 -> 2026-02-22
Running in MARKET-ONLY mode - using Gamma API as game catalog (no NBA stats required)
Found 925 Gamma markets
Built game catalog with 925 games
Built timeline with 925 entries
Market lookup has 925 entries
Matched 925 markets to games


,game_date,game_id,home_team,away_team,market_id,event_title,question,yes_team_side
0,2026-02-20 00:00:00+00:00,GAMMA_1402321,Unknown,Unknown,1402321,Wizards vs. Hawks,Wizards vs. Hawks,home
1,2026-02-20 00:00:00+00:00,GAMMA_1440482,Unknown,Unknown,1440482,Wizards vs. Hawks,Spread: Hawks (-11.5),home
2,2026-02-20 00:00:00+00:00,GAMMA_1440483,Unknown,Unknown,1440483,Wizards vs. Hawks,Wizards vs. Hawks: O/U 233.5,home
3,2026-02-20 00:00:00+00:00,GAMMA_1440484,Unknown,Unknown,1440484,Wizards vs. Hawks,Spread: Hawks (-12.5),home
4,2026-02-20 00:00:00+00:00,GAMMA_1441779,Unknown,Unknown,1441779,Wizards vs. Hawks,Wizards vs. Hawks: O/U 234.5,home
5,2026-02-20 00:00:00+00:00,GAMMA_1442179,Unknown,Unknown,1442179,Wizards vs. Hawks,Jalen Johnson: Points O/U 22.5,home
6,2026-02-20 00:00:00+00:00,GAMMA_1442180,Unknown,Unknown,1442180,Wizards vs. Hawks,CJ McCollum: Points O/U 19.5,home
7,2026-02-20 00:00:00+00:00,GAMMA_1442181,Unknown,Unknown,1442181,Wizards vs. Hawks,Nickeil Alexander-Walker: Points O/U 19.5,home
8,2026-02-20 00:00:00+00:00,GAMMA_1442182,Unknown,Unknown,1442182,Wizards vs. Hawks,Onyeka Okongwu: Points O/U 16.5,home
9,2026-02-20 00:00:00+00:00,GAMMA_1442183,Unknown,Unknown,1442183,Wizards vs. Hawks,Tristan Vukcevic: Points O/U 12.5,home


Beginning Backtest Iteration on 32 files


ValueError: right keys must be sorted